# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [1]:
# Gemini import
from openai import OpenAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [2]:
import os, re, pathlib, requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
!pip install -q faiss-cpu
import faiss

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# TODO (you): replace with your own technical-doc corpus. This placeholder is one short doc. I chose to cover pololu remote control cars, I was originally going to do arduino since
# I have been working on arduino remote control cars lately for a robotics class, however those documents wouldn't scrape properly due to formatting.
DOC_URLS = {
    'astar_32u4': 'https://www.pololu.com/docs/0J61/all',
    'astar_robot_controller': 'https://www.pololu.com/docs/0J66/all',
    'zumo_32u4': 'https://www.pololu.com/docs/0J63/all',
    'zumo_shield': 'https://www.pololu.com/docs/0J57',
    'three_pi_plus': 'https://www.pololu.com/docs/0J83/all',
    'zumo_chassis': 'https://www.pololu.com/docs/0J54/all',
    'qtr_sensors': 'https://www.pololu.com/docs/0J19/all',
    'mc33926_motor_shield': 'https://www.pololu.com/docs/0j55/all'
}

def load_doc(url):
    response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    chunks = []
    for element in soup.select("h1, h2, h3, h4, p, li"):
        text = re.sub(r"\s+", " ", element.get_text(" ", strip=True)).strip()
        if text:
            chunks.append(text)
    return "\n\n".join(chunks)

DOCS = {}
for name, url in DOC_URLS.items():
    try:
        DOCS[name] = load_doc(url)
        print(f"{name:22s} -> {len(DOCS[name]):,} characters")
    except requests.RequestException as e:
        print(f"{name:22s} -> FAILED: {e}")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

astar_32u4             -> 81,327 characters
astar_robot_controller -> 69,057 characters
zumo_32u4              -> 101,830 characters
zumo_shield            -> 1,216 characters
three_pi_plus          -> 91,048 characters
zumo_chassis           -> 8,518 characters
qtr_sensors            -> 26,947 characters
mc33926_motor_shield   -> 32,927 characters


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += size - overlap

    return chunks

def chunk_paragraph(text):
    return [
        re.sub(r"\s+", " ", paragraph).strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()
    ]

configs = {
    "small_120_20": {name: chunk_fixed(text, 120, 20)
        for name, text in DOCS.items()
    },
    "large_320_40": {name: chunk_fixed(text, 320, 40)
        for name, text in DOCS.items()
    },
    "paragraph": {name: chunk_paragraph(text)
        for name, text in DOCS.items()
    }
}

for config_name, documents in configs.items():
    total = sum(len(chunks) for chunks in documents.values())
    print(f"{config_name:16s} -> {total:,} chunks")

    for name, chunks in documents.items():
        print(f"    {name:25s}: {len(chunks)}")


small_120_20     -> 4,104 chunks
    astar_32u4               : 809
    astar_robot_controller   : 687
    zumo_32u4                : 1012
    zumo_shield              : 12
    three_pi_plus            : 905
    zumo_chassis             : 84
    qtr_sensors              : 268
    mc33926_motor_shield     : 327
large_320_40     -> 1,468 chunks
    astar_32u4               : 289
    astar_robot_controller   : 246
    zumo_32u4                : 362
    zumo_shield              : 4
    three_pi_plus            : 324
    zumo_chassis             : 30
    qtr_sensors              : 96
    mc33926_motor_shield     : 117
paragraph        -> 2,650 chunks
    astar_32u4               : 488
    astar_robot_controller   : 417
    zumo_32u4                : 611
    zumo_shield              : 66
    three_pi_plus            : 547
    zumo_chassis             : 117
    qtr_sensors              : 193
    mc33926_motor_shield     : 211


In [4]:
# Prepare each chunking configuration for FAISS retrieval.
# Each chunk keeps data identifying which document it came from.

prepared = {}

for config_name, documents in configs.items():
    all_chunks = []
    metadata = []

    for document_name, chunks in documents.items():
        for chunk_number, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            metadata.append({
                "document": document_name,
                "chunk_number": chunk_number
            })

    prepared[config_name] = {
        "chunks": all_chunks,
        "metadata": metadata
    }

    print(
        f"{config_name:16s} -> "
        f"{len(all_chunks):,} chunks"
    )


def build_index(chunks):
    embeddings = embedder.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index


# Build one FAISS index for each chunking configuration.
indices = {}

for config_name, data in prepared.items():
    indices[config_name] = build_index(data["chunks"])

    print(
        f"{config_name}: "
        f"FAISS index contains {indices[config_name].ntotal} vectors"
    )


def retrieve(config_name, query, k=3):
    data = prepared[config_name]
    index = indices[config_name]

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = index.search(query_embedding, k)

    results = []

    for score, position in zip(scores[0], positions[0]):
        if position >= 0:
            results.append({
                "score": float(score),
                "text": data["chunks"][position],
                "document": data["metadata"][position]["document"],
                "chunk_number": data["metadata"][position]["chunk_number"]
            })

    return results



small_120_20     -> 4,104 chunks
large_320_40     -> 1,468 chunks
paragraph        -> 2,650 chunks


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

small_120_20: FAISS index contains 4104 vectors


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

large_320_40: FAISS index contains 1468 vectors


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

paragraph: FAISS index contains 2650 vectors


In [5]:
def retrieve(config_name, query, k=3):
    data = prepared[config_name]
    index = indices[config_name]

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = index.search(query_embedding, k)

    results = []

    for score, position in zip(scores[0], positions[0]):
        if position >= 0:
            results.append({
                "score": float(score),
                "text": data["chunks"][position],
                "document": data["metadata"][position]["document"],
                "chunk_number": data["metadata"][position]["chunk_number"]
            })

    return results


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [6]:
import numpy as np
qa = [
    ("What microcontroller does the A-Star 32U4 use?", "astar_32u4", ("ATmega32U4",)),
    ("What type of motor driver is used by the Zumo 32U4?", "zumo_32u4", ("DRV8838",)),
    ("How are the motors controlled on the Zumo 32U4?", "zumo_32u4", ("PWM",)),
    ("What voltage range can the Zumo 32U4 operate from?", "zumo_32u4", ("2.7 V to 10 V",)),
    ("What is the purpose of the encoders on the Zumo 32U4?", "zumo_32u4", ("encoder",)),
    ("What does the MC33926 motor driver shield control?", "mc33926_motor_shield", ("DC motors",)),
    ("How does PWM control motor speed?", "zumo_32u4", ("PWM",)),
    ("What sensors are included on the Zumo 32U4?", "zumo_32u4", ("sensors",)),
    ("What is the purpose of the QTR reflectance sensors?", "qtr_sensors", ("reflectance",)),
    ("What is the Zumo Shield designed to work with?", "zumo_shield", ("Arduino",)),
    ("What kind of motors are used with the Zumo chassis?", "zumo_chassis", ("motors",)),
    ("What programming environment can be used with the A-Star 32U4?", "astar_32u4", ("Arduino IDE",)),
]

norm = lambda s: re.sub(r"\s+", " ", s).lower()

# Label relevant chunks in advance.
labels = {
    name: [
        {i for i, (doc, text) in enumerate(
            zip(prepared[name]["metadata"], prepared[name]["chunks"]))
          if doc["document"] == document and any(
            norm(fact) in norm(text) for fact in facts)}
        for _, document, facts in qa]
    for name in configs
}
K = 3
print(f'{"config":16s} {"precision@3":>12s} {"recall@3":>10s}')

for name in configs:
    retrieved = [set((r["document"], r["chunk_number"])
            for r in retrieve(name, question, k=K))
        for question, _, _ in qa
    ]

    relevant = [set((prepared[name]["metadata"][i]["document"],
             prepared[name]["metadata"][i]["chunk_number"])
            for i in label_set)
        for label_set in labels[name]]

    precision = np.mean([
        len(r & rel) / K
        for r, rel in zip(retrieved, relevant)])

    recall = np.mean([
        len(r & rel) / len(rel) if rel else 0
        for r, rel in zip(retrieved, relevant)])
    print(f"{name:16s} {precision:12.2f} {recall:10.2f}")


config            precision@3   recall@3
small_120_20             0.36       0.07
large_320_40             0.39       0.18
paragraph                0.31       0.18


In [7]:
# Showing conflicting chunks
for question, _, facts in qa:
    results_by_config = {}

    for config_name in configs:
        results = retrieve(config_name, question, k=3)
        relevant = sum(1
            for result in results
            if any(norm(fact) in norm(result["text"]) for fact in facts))
        results_by_config[config_name] = relevant

    if len(set(results_by_config.values())) > 1:
        print(f"\nQuestion: {question}")

        for config_name, relevant in results_by_config.items():
            print(f"  {config_name}: {relevant}/3 relevant chunks")



Question: What microcontroller does the A-Star 32U4 use?
  small_120_20: 1/3 relevant chunks
  large_320_40: 1/3 relevant chunks
  paragraph: 0/3 relevant chunks

Question: What type of motor driver is used by the Zumo 32U4?
  small_120_20: 2/3 relevant chunks
  large_320_40: 0/3 relevant chunks
  paragraph: 1/3 relevant chunks

Question: What voltage range can the Zumo 32U4 operate from?
  small_120_20: 0/3 relevant chunks
  large_320_40: 1/3 relevant chunks
  paragraph: 1/3 relevant chunks

Question: What is the purpose of the encoders on the Zumo 32U4?
  small_120_20: 3/3 relevant chunks
  large_320_40: 3/3 relevant chunks
  paragraph: 2/3 relevant chunks

Question: What sensors are included on the Zumo 32U4?
  small_120_20: 3/3 relevant chunks
  large_320_40: 3/3 relevant chunks
  paragraph: 1/3 relevant chunks

Question: What is the Zumo Shield designed to work with?
  small_120_20: 2/3 relevant chunks
  large_320_40: 3/3 relevant chunks
  paragraph: 3/3 relevant chunks

Question

In [8]:
# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.

# The different chunking configurations disagree because each one breaks up the information in the documents differently, which can affect what information gets pulled when
# searching for an answer. The larger chunks generally seem to do better because they include more surrounding information, which gives the retrieval system more context
# to identify the correct answer. The smaller chunks can sometimes be more specific, but they can also separate the answer from the information that gives it context,
# which can cause the wrong chunk to be retrieved. The paragraph configuration seems to have mixed results as well, since some answers are contained clearly within
# individual paragraphs while other information can be split between multiple sections. I think this shows that there is a trade off between having smaller chunks that
# are more specific and larger chunks that contain more information. Overall, the results show that changing the chunk size can have a pretty significant effect on which
# information is retrieved, even when the same questions and documents are being used.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [9]:
def gemini_chat(messages, model="gemini-3.6-flash"):
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content


def answer_question(question, config_name="paragraph", k=3):
    retrieved = retrieve(config_name, question, k=k)
    context = "\n\n".join(
        f"[Source: {r['document']}]\n{r['text']}"
        for r in retrieved
    )

    prompt = f"""Answer the question using only the provided technical documentation.
If the documentation does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
{context}

Question:
{question}

Answer:"""

    return gemini_chat([
        {"role": "user", "content": prompt}
    ])



In [10]:
answer_question("What microcontroller does the A-Star 32U4 use?") #Failure

'The information is not available in the provided context.'

In [11]:
answer_question("What type of motor driver is used by the Zumo 32U4?")

'Based on the provided documentation, the Zumo 32U4 uses Texas Instruments DRV8838 motor drivers.'

In [12]:
answer_question("How are the motors controlled on the Zumo 32U4?")

'Based on the provided technical documentation, the motors on the Zumo 32U4 are controlled through the following:\n\n* **Hardware:** Two on-board Texas Instruments DRV8838 motor drivers power the motors, and four Arduino pins are used to control these drivers.\n* **Software:** The `Zumo32U4` library provides functions to easily control the motors. It can also optionally flip a direction signal if a motor was soldered in backwards.'

In [13]:
answer_question("What voltage range can the Zumo 32U4 operate from?")

'Based on the provided documentation, the Zumo 32U4 can accept a battery input voltage from 2.7 V to 10 V. The maximum allowable voltage can be raised to 11 V by disconnecting or modifying the battery voltage divider.'

In [14]:
#Part 4: Failure
''' The question "What microcontroller does the A-Star 32U4 use?" failed because it could not find an answer in the chunks.
The correct answer is the ATmega32U4, however the paragraph chunking configuration did not retrieve any of the chunks containing that information in the top three results.
Because of this, the generation model did not have the correct information available and instead said that the information was not available in the provided context.
I think this is a good example of one of the biggest problems with RAG systems, where even if the information exists somewhere in the documentation,
the AI cannot necessarily give the correct answer because the retrieval system might fail at finding it. One way to mitigate this would be using a different chunking configuration,
such as the larger chunks, or increasing the number of chunks retrieved so there is a better chance of finding the correct section. This would give the AI a
larger pool of relevant information to work with while still keeping the response grounded in the documentation.
'''


' The question "What microcontroller does the A-Star 32U4 use?" failed because it could not find an answer in the chunks. \nThe correct answer is the ATmega32U4, however the paragraph chunking configuration did not retrieve any of the chunks containing that information in the top three results. \nBecause of this, the generation model did not have the correct information available and instead said that the information was not available in the provided context. \nI think this is a good example of one of the biggest problems with RAG systems, where even if the information exists somewhere in the documentation, \nthe AI cannot necessarily give the correct answer because the retrieval system might fail at finding it. One way to mitigate this would be using a different chunking configuration, \nsuch as the larger chunks, or increasing the number of chunks retrieved so there is a better chance of finding the correct section. This would give the AI a \nlarger pool of relevant information to wo

## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.